In [ ]:
import spacy
# from spacy.tokens import Doc

In [ ]:
nlp = spacy.load('en_core_web_sm')

# print(nlp.pipe_names)

In [ ]:
doc = nlp("Sales data. Sagar need sales data. Current february month sales information.")
doc2 = nlp("Need sales data for current month.")

# for vocabulary in nlp.vocab:
#     print(vocabulary.text)

# for entity in doc.ents:
#     print(entity.text, entity.label_)

# for sent in doc.sents:
#     score = sent.similarity(doc2)
#     print("SENT::", sent.text, score)

#     sent_doc = nlp(sent.text)
#     print("SENT::", sent.text, sent_doc.cats)

# for token in doc:
#     print(token.text, token.pos_, token.lemma_, token.is_sent_start, token.is_sent_end)

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

pattern = [{"POS": "NOUN", "OP": "+"}]
matcher.add("CUSTOM_NOUN", [pattern], greedy="LONGEST")

matches = matcher(doc)

for match in matches:
    print(nlp.vocab[matches[0][0]].text, doc[match[1]: match[2]])

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

patterns = {
    "SALES_DATA_CUR": [
        [
            {"LEMMA": "sale"},
            {"LEMMA": {"IN": ["datum", "information"]}},
            {"LOWER": {"IN": ["for", "of"]}, "OP": "?"},
            {"LOWER": {"IN": ["current", "this"]}},
            {"LEMMA": "month"}
        ],
        [
            {"LOWER": {"IN": ["current", "this"]}}, 
            {"LEMMA": "month"},
            {"LEMMA": "sale"},
            {"LEMMA": {"IN": ["datum", "information"]}},
        ]
    ],

    "SALES_DATA_PREV": [
        [
            {"LEMMA": "sale"},
            {"LEMMA": {"IN": ["datum", "information"]}},
            {"LOWER": {"IN": ["for", "of"]}, "OP": "?"},
            {"LOWER": {"IN": ["previous", "last"]}},
            {"LEMMA": "month"}
        ]
    ],

    "SALES_DATA_PREV_N": [
        [
            {"LEMMA": "sale"},
            {"LEMMA": {"IN": ["datum", "information"]}},
            {"LOWER": {"IN": ["for", "of"]}, "OP": "?"},
            {"LOWER": {"IN": ["previous", "last"]}},
            {"IS_DIGIT": True},
            {"LEMMA": "month"}
        ]
    ],
    "SALES_DATA_MTM": [
        [
            {"LEMMA": "sale"},
            {"LEMMA": {"IN": ["datum", "information"]}},
            {"LOWER": {"IN": ["for", "of", "from"]}, "OP": "?"},
            {"ENT_TYPE": "DATE"}
        ]
    ]
}

for intent, pattern in patterns.items():
    matcher.add(intent, pattern)

docs = [
    "Show me sales data for current month",
    "I need sales data for last 2 months",
    "Give me sales data of previous month",
    "Current month sales data",
    "Sales data from february to june"
]

for doc in docs:
    nlp_doc = nlp(doc)

    for ent in nlp_doc.ents:
        print("ENTITY:: ", ent.text, ent.label_)
        
    matches = matcher(nlp_doc)
    for match_id, start, end in matches:
        
        for token in nlp_doc[start:end]:
            print('TOKEN:: ', token)

        print(
            f"{nlp.vocab.strings[match_id]} --> "
            f"{nlp_doc[start:end].text}"
        )



In [ ]:

from spacy.matcher import PhraseMatcher

p_matcher = PhraseMatcher(nlp.vocab)
p_matcher.add("OBAMA", [nlp("Barack Obama")])
doc_pm = nlp("Barack Obama lifts America one last time in emotional farewell")
matches_oo = p_matcher(doc_pm)

for match_id, start, end in matches_oo:
    print(
        f"{nlp.vocab.strings[match_id]} --> "
        f"{doc_pm[start:end].text}"
    )

In [ ]:
# for doc in docs:
#     nlp_doc = nlp(doc)

#     # Remove stop words
#     filtered_texts = [token.text for token in nlp_doc if not token.is_stop]
#     filtered_doc = nlp(' '.join(filtered_texts))

#     matches = matcher(filtered_doc)

#     for match_id, start, end in matches:
#         print(
#             f"{nlp.vocab.strings[match_id]} --> "
#             f"{filtered_doc[start:end].text}"
#         )

In [ ]:
from spacy.language import Language

@Language.component('remove_gpe')
def remove_gpe(doc):
    original_ents = list(doc.ents)
    for ent in doc.ents:
        if ent.label == 'GPE':
            original_ents.remove(ent)
    
    doc.ents = original_ents
    return (doc)

In [ ]:
nlp = spacy.load('en_core_web_sm')
matcher = Matcher(nlp.vocab)

matcher.add('SALES_DATA', [[{"LOWER": "sales"}]])

doc = "current month sales data"
nlp_doc = nlp(doc)

matches = matcher(nlp_doc)

for match in matches:
    print(nlp.vocab[match[0]].text)
